# Lab 1 and 2 (Day 1)

In [10]:
from pathlib import Path
import os, sys, subprocess, importlib.metadata
IS_COLAB = Path('/content').is_dir() and 'google.colab' in sys.modules
if IS_COLAB:
    ROOT = Path('/content/masar-modern-data-engineering')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/almiyead-rgb/masar-modern-data-engineering.git',str(ROOT)], check=True)
    pins = {'pyspark':'3.5.8','delta-spark':'3.3.3','py4j':'0.10.9.9'}
    missing = []
    for package, version in pins.items():
        try: observed = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: observed = None
        if observed != version: missing.append(f'{package}=={version}')
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
    candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    if not any((p/'bin/java').is_file() for p in candidates):
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
        candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    os.environ['JAVA_HOME'] = str(next(p for p in candidates if (p/'bin/java').is_file()))
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.json').is_file()),None)
    if ROOT is None: raise FileNotFoundError('Open from the repository root; see docs/SETUP.md')
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])


Repository: /content/masar-modern-data-engineering
Python: 3.11.13


In [11]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


In [12]:
from masar.sources import verify_manifest, load_sources, profile_sources
SOURCE = ROOT / "data" / "masar-small-v1"
manifest = verify_manifest(SOURCE)
feeds = load_sources(SOURCE)
print("Dataset:", manifest["label"])
print("Verified manifest files:", len(manifest["files"]))
print(json.dumps({name: len(rows) for name, rows in feeds.items()}, indent=2))
print("Trip columns:", list(feeds["trips"][0]))
print("First synthetic event:", json.dumps(feeds["gps_events"][0], ensure_ascii=False))

Dataset: MASAR_SMALL_V1
Verified manifest files: 10
{
  "trips": 72,
  "drivers": 6,
  "gps_events": 216
}
Trip columns: ['trip_id', 'driver_id', 'city', 'start_ts', 'end_ts', 'fare_sar', 'distance_km']
First synthetic event: {"city": "Riyadh", "event_id": "SYN_E0001_0", "event_ts": "2026-06-01T06:00:00+03:00", "location": {"lat": 24.7, "lon": 46.7}, "synthetic": true, "trip_id": "SYN_T0001"}


In [13]:
result = profile_sources(SOURCE)
print(json.dumps(result["profile"], indent=2))
print(json.dumps(result["relations"], indent=2))
print(json.dumps(result["city_profile"], indent=2))

{
  "trips": {
    "rows": 72,
    "key": "trip_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "distance_km": 0,
      "driver_id": 0,
      "end_ts": 0,
      "fare_sar": 0,
      "start_ts": 0,
      "trip_id": 0
    }
  },
  "drivers": {
    "rows": 6,
    "key": "driver_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "driver_id": 0,
      "driver_rating": 0,
      "vehicle_type": 0
    }
  },
  "gps_events": {
    "rows": 216,
    "key": "event_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "event_id": 0,
      "event_ts": 0,
      "location": 0,
      "synthetic": 0,
      "trip_id": 0
    }
  }
}
{
  "trips_without_driver": 0,
  "events_without_trip": 0,
  "events_with_invalid_coordinates": 0
}
{
  "original_labels": {
    " dammam ": 3,
    " jeddah ": 3,
    " riyad

In [14]:
if not all(result["checks"].values()):
    raise AssertionError(result["checks"])
output = RUN / "source_inspection.json"
output.write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print(json.dumps(result["checks"], indent=2))
print("PASS: source inspection only")
print("Saved:", output.name)
print("Source inspection complete. Continue to the Bronze section.")

{
  "source_counts": true,
  "unique_base_keys": true,
  "base_top_level_complete": true,
  "valid_links_and_coordinates": true,
  "three_events_per_trip": true,
  "base_city_set": true
}
PASS: source inspection only
Saved: source_inspection.json
Source inspection complete. Continue to the Bronze section.


In [15]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.13",
  "java": "openjdk version \"17.0.20\" 2026-07-21",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


In [16]:
require_environment()
from masar.workspace import new_workspace, require_fixed_dataset, write_json, workspace_path
require_fixed_dataset(SOURCE)
WORK = new_workspace(ROOT, "day01_bronze")
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_lo_ncbvo


In [17]:
from masar.bronze import raw_frame, ingest_feed, verify_bronze
# Functions are imported without starting Spark. The next cell executes the lab.
print("Bronze functions loaded. The next cell writes and reads the Delta tables.")

Bronze functions loaded. The next cell writes and reads the Delta tables.


In [18]:
from masar.workspace import record_bronze_success
spark = start_spark(WORK)
try:
    print("Spark:", spark.version)
    raw_trips = raw_frame(spark, SOURCE, "trips")
    raw_trips.printSchema()
    raw_trips.show(3, truncate=False)
    print("Source trips:", raw_trips.count())
    for feed in ("trips", "drivers", "gps_events"):
        print(ingest_feed(spark, SOURCE, WORK, feed, "base_001"))
    print(ingest_feed(spark, SOURCE, WORK, "trips", "replay_002"))
    report = verify_bronze(spark, SOURCE, WORK)
    record_bronze_success(ROOT, WORK)
    print(json.dumps({"counts": report["counts"], "checks": report["checks"]}, indent=2))
    print("Retained:", (WORK / "reports/bronze.json").relative_to(ROOT))
finally:
    spark.stop()

Spark: 3.5.8
root
 |-- trip_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- start_ts: string (nullable = true)
 |-- end_ts: string (nullable = true)
 |-- fare_sar: string (nullable = true)
 |-- distance_km: string (nullable = true)

+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|trip_id  |driver_id|city  |start_ts                 |end_ts                   |fare_sar|distance_km|
+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|SYN_T0001|SYN_D001 |Riyadh|2026-06-01T06:00:00+03:00|2026-06-01T06:08:00+03:00|18.00   |3.50       |
|SYN_T0002|SYN_D002 |Riyadh|2026-06-01T08:00:00+03:00|2026-06-01T08:11:00+03:00|19.25   |3.85       |
|SYN_T0003|SYN_D001 |Riyadh|2026-06-01T10:00:00+03:00|2026-06-01T10:14:00+03:00|20.50   |4.20       |
+---------+---------+------+-------------------------+-------------------------+--------+---

In [19]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


In [20]:
from decimal import Decimal
from masar.cost import DEFAULTS, evaluate_cost, serializable, cost_report
print(json.dumps(DEFAULTS, indent=2))
result = cost_report()
print(json.dumps(result["base"], indent=2))

{
  "days": "30",
  "hours_per_day": "24",
  "cores": "4",
  "price_per_core_hour": "0.50",
  "storage_gib": "100",
  "storage_price_per_gib_month": "0.20",
  "work_hours_per_day": "2",
  "startup_hours_per_day": "0.25",
  "scheduled_extra_monthly": "30",
  "always_on_extra_monthly": "0"
}
{
  "unit": "TU (hypothetical teaching units; not currency)",
  "always_on_hours": "720",
  "scheduled_hours": "67.50",
  "always_on_compute": "1440.00",
  "scheduled_compute": "135.0000",
  "storage_each": "20.00",
  "always_on_total": "1460.00",
  "scheduled_total": "185.0000",
  "difference": "1275.0000",
  "reduction_fraction": "0.8732876712328767123287671233",
  "break_even_work_hours_per_day": "23.25"
}


In [21]:
print("Work h/day | Always-on TU | Scheduled TU | Difference TU")
for row in result["sensitivity"]:
    print(f"{row['work_hours_per_day']:>10} | {row['always_on_total']:>12} | {row['scheduled_total']:>12} | {row['difference']:>13}")

Work h/day | Always-on TU | Scheduled TU | Difference TU
         0 |      1460.00 |        50.00 |       1410.00
         1 |      1460.00 |     125.0000 |     1335.0000
         2 |      1460.00 |     185.0000 |     1275.0000
         4 |      1460.00 |     305.0000 |     1155.0000
         8 |      1460.00 |     545.0000 |      915.0000
        12 |      1460.00 |     785.0000 |      675.0000
        18 |      1460.00 |    1145.0000 |      315.0000
        23 |      1460.00 |    1445.0000 |       15.0000
     23.25 |      1460.00 |    1460.0000 |        0.0000
     23.75 |      1460.00 |    1490.0000 |      -30.0000


In [22]:
base = evaluate_cost(DEFAULTS)
checks = {
    "base_totals": (base["always_on_total"], base["scheduled_total"]) == (Decimal("1460"), Decimal("185")),
    "break_even": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.25"})["difference"] == 0,
    "counterexample": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.75"})["difference"] < 0,
    "zero_rate": evaluate_cost({**DEFAULTS, "price_per_core_hour": "0"})["break_even_work_hours_per_day"] is None,
}
try:
    evaluate_cost({**DEFAULTS, "cores": "-1"})
except ValueError:
    checks["negative_input_rejected"] = True
else:
    checks["negative_input_rejected"] = False
if not all(checks.values()):
    raise AssertionError(checks)
result["checks"] = checks
print(json.dumps(checks, indent=2))

{
  "base_totals": true,
  "break_even": true,
  "counterexample": true,
  "zero_rate": true,
  "negative_input_rejected": true
}


In [23]:
output = RUN / "cost_model_result.json"
output.write_text(json.dumps(result, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("PASS: hypothetical cost arithmetic only")
print("Saved:", output.name)
print("Cost arithmetic complete. Continue to the measured Spark comparison.")

PASS: hypothetical cost arithmetic only
Saved: cost_model_result.json
Cost arithmetic complete. Continue to the measured Spark comparison.


In [24]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.13",
  "java": "openjdk version \"17.0.20\" 2026-07-21",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


In [25]:
require_environment()
from masar.workspace import completed_bronze_workspace, require_fixed_dataset
require_fixed_dataset(SOURCE)
WORK = completed_bronze_workspace(ROOT)
spark = start_spark(WORK)
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_lo_ncbvo


In [26]:
from masar.benchmark import benchmark
try:
    report = benchmark(spark, SOURCE, WORK, repetitions=4)
    print(json.dumps(report["expected_and_observed_aggregate"], indent=2))
    print(json.dumps(report["measurements"], indent=2))
    print("Plans:", report["plans"])
    print("Evidence:", (WORK / "reports/benchmark.json").relative_to(ROOT))
finally:
    spark.stop()

{
  "rows": 72,
  "nonnull_fares": 72,
  "fare_total": "1794.60"
}
{
  "csv": {
    "samples_s": [
      0.48706269699869154,
      0.3742991130002338,
      0.40085146100136626,
      0.4916396230000828
    ],
    "median_s": 0.4439570790000289,
    "min_s": 0.3742991130002338,
    "max_s": 0.4916396230000828
  },
  "delta_v0": {
    "samples_s": [
      3.6888499910000974,
      3.1596716520016344,
      2.6092098009994515,
      2.627146773000277
    ],
    "median_s": 2.8934092125009556,
    "min_s": 2.6092098009994515,
    "max_s": 3.6888499910000974
  }
}
Plans: {'csv': 'reports/plans/csv.txt', 'delta_v0': 'reports/plans/delta_v0.txt'}
Evidence: outputs/day01_bronze_lo_ncbvo/reports/benchmark.json


In [27]:
# DAY01_HANDOFF_V2: retain the pointer and all small reports as well as Delta files.
from pathlib import Path
import zipfile
from masar.workspace import completed_bronze_workspace
WORK = completed_bronze_workspace(ROOT)
pointer = ROOT / 'outputs/day01_bronze_success.json'
files_to_save = {pointer, *(p for p in WORK.rglob('*') if p.is_file())}
for name in ('source_inspection.json', 'cost_model_result.json'):
    files_to_save.update((ROOT / 'outputs').rglob(name))
archive = ROOT / 'outputs/day01_handoff.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(files_to_save):
        bundle.write(path, arcname=path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    assert bundle.read('outputs/day01_bronze_success.json') == pointer.read_bytes()
    assert any(name.endswith('source_inspection.json') for name in bundle.namelist())
    assert any(name.endswith('cost_model_result.json') for name in bundle.namelist())
print('Keep this ZIP for the next day:', archive)
print('Also save this notebook with outputs and your LAB01/LAB02 notes.')
if IS_COLAB:
    from google.colab import files
    files.download(str(archive))

Keep this ZIP for the next day: /content/masar-modern-data-engineering/outputs/day01_handoff.zip
Also save this notebook with outputs and your LAB01/LAB02 notes.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Lab 3 (Day2)

In [33]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_lo_ncbvo


In [34]:
from masar.silver import run_staging_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_staging_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03a_staging', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Observed staging row counts:', result['counts'])
    preview = WORK / ('mini_lakehouse/staging/day02_' + result['run_id'] + '/stg_trips')
    spark.read.format('delta').load(str(preview)).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY02_STAGING_ENGINE",
  "checks": {
    "counts_verified": true,
    "typed_values_match_source_oracle": true,
    "drivers_unique_and_join_safe": true,
    "gps_valid": true,
    "delta_readback": true
  }
}
Observed staging row counts: {'stg_trips': 144, 'stg_drivers': 6, 'stg_gps': 216}
+---------+------+--------+
|trip_id  |city  |fare_sar|
+---------+------+--------+
|SYN_T0001|Riyadh|18.00   |
|SYN_T0001|Riyadh|18.00   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0003|Riyadh|20.50   |
+---------+------+--------+
only showing top 5 rows



In [35]:
from masar.silver import run_incremental_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_incremental_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03b_silver', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    spark.read.format('delta').load(str(WORK/'mini_lakehouse/silver/trips')).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY02_SILVER_ENGINE",
  "checks": {
    "actual_delta_files": true,
    "all_scenarios_match_independent_oracle": true,
    "business_keys_unique": true,
    "late_rows_retained": true,
    "replay_preserves_business_content": true
  }
}
+-----------+------+--------+
|trip_id    |city  |fare_sar|
+-----------+------+--------+
|SYN_LATE001|Riyadh|25.00   |
|SYN_LATE002|Jeddah|27.00   |
|SYN_LATE003|Dammam|29.00   |
|SYN_T0001  |Riyadh|18.00   |
|SYN_T0002  |Riyadh|19.25   |
+-----------+------+--------+
only showing top 5 rows



In [23]:
%pip install -q dbt-core==1.9.8 dbt-spark==1.9.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.0/947.0 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.9/144.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.0/167.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.7/442.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [36]:
from masar.dbt_lab import run_dbt_lab
dbt_report, dbt_path = run_dbt_lab(ROOT)
print(json.dumps({'status': dbt_report['status'], 'phases_completed': len(dbt_report['phases']), 'report': str(dbt_path.relative_to(ROOT)), 'error': dbt_report.get('error')}, indent=2))
assert dbt_report['status'] == 'PASSED_DBT_NATIVE', dbt_report.get('error')

# Observed learning output
for phase in dbt_report['phases']:
    print(phase['phase'], 'rows:', phase['rows'], 'fare SAR:', phase['total_fare_sar'])
print('Catalog evidence:', dbt_report['commands'][-1])

{
  "status": "PASSED_DBT_NATIVE",
  "phases_completed": 4,
  "report": "outputs/dbt_validation_q0bblj7d/reports/dbt_attempt.json",
  "error": null
}
base rows: 72 fare SAR: 1794.60
rerun rows: 72 fare SAR: 1794.60
late rows: 75 fare SAR: 1875.60
late_replay rows: 75 fare SAR: 1875.60
Catalog evidence: {'catalog_sha256': 'aa327f87dd99d4f78fc92150fda00dc601bb0929fcc626f8c75d40d55b0e5165', 'models_documented': 6, 'sources_documented': 3, 'metadata_method': 'native DESCRIBE TABLE EXTENDED', 'phase': 'documentation', 'command': ['docs', 'generate'], 'target': 'dbt/commands/documentation/target'}


In [37]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day02_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    dbt_workspace = dbt_path.parent.parent
    for p in sorted(dbt_workspace.rglob('*')):
        if p.is_file():
            bundle.write(p, p.relative_to(ROOT).as_posix())
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day02_handoff.zip


# Summary to understand flow pipeline


  
               SOURCE SYSTEM
                     │
                     │
          CSV (raw files)
                     │
                     ▼
           ┌─────────────────┐
           │     PySpark     │
           │ processing tool │
           └────────┬────────┘
                    │ writes
                    ▼
        ┌───────────────────────┐
        │     BRONZE DELTA      │
        │                       │
        │ Raw deliveries        │
        │ Preserve source       │
        │ Replay allowed        │
        └───────────┬───────────┘
                    │
              PySpark cleans
                    │
                    ▼
        ┌───────────────────────┐
        │       STAGING         │
        │                       │
        │ Correct data types    │
        │ Normalize fields      │
        │ Prepare for business  │
        │ logic                 │
        └───────────┬───────────┘
                    │
        PySpark/business rules
                    │
                    ▼
        ┌───────────────────────┐
        │      SILVER DELTA     │
        │                       │
        │ Deduplicated          │
        │ Clean                 │
        │ Incremental           │
        │ Business trustworthy  │
        └───────────┬───────────┘
                    │
                    │ dbt models
                    ▼
          ┌──────────────────┐
          │       dbt        │
          │                  │
          │ SQL models       │
          │ Tests            │
          │ Incremental logic│
          │ Documentation    │
          └────────┬─────────┘
                   │
                   ▼
        ┌─────────────────────┐
        │ Final data models   │
        │                     │
        │ Analytics           │
        │ Dashboards          │
        │ Reporting           │
        └─────────────────────┘

# Lab 4 (Day 3)

In [38]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_lo_ncbvo


In [39]:
from masar.delta_lab import run_transactions_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_transactions_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04a_transactions', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({k: result[k] for k in ('before','after_correction','past_version_read')}, indent=2, default=str))
finally:
    spark.stop()

{
  "scope": "DAY03_TRANSACTIONS_ENGINE",
  "checks": {
    "native_correction_matches_source_expectation": true,
    "business_rows_stay_75": true,
    "replay_and_stale_delivery_preserve_values": true,
    "same_revision_conflict_rejected": true,
    "actual_prior_version_read": true,
    "mixed_valid_invalid_batch_rejected_atomically": true
  }
}
{
  "before": {
    "rows": 75,
    "business_digest": "0d16e2795620ae0c0f54d3fcd52a5b13fb2c2a47cd4d0c42c8ddb195f19bc6e0",
    "version": 1,
    "schema": [
      [
        "trip_id",
        "string"
      ],
      [
        "driver_id",
        "string"
      ],
      [
        "city",
        "string"
      ],
      [
        "start_utc",
        "timestamp"
      ],
      [
        "end_utc",
        "timestamp"
      ],
      [
        "trip_date_local",
        "date"
      ],
      [
        "fare_sar",
        "decimal(12,2)"
      ],
      [
        "distance_km",
        "decimal(12,2)"
      ],
      [
        "duration_seconds",

In [40]:
from masar.delta_lab import run_maintenance_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_maintenance_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04b_maintenance', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({'recovery': result['recovery'], 'vacuum': result['vacuum']}, indent=2, default=str))
finally:
    spark.stop()

{
  "scope": "DAY03_MAINTENANCE_ENGINE",
  "checks": {
    "unexpected_column_rejected": true,
    "approved_evolution_preserves_business_values": true,
    "compaction_preserves_values": true,
    "delete_affects_copy_only": true,
    "restore_creates_new_commit": true,
    "vacuum_is_non_destructive_dry_run": true,
    "trusted_silver_unchanged": true
  }
}
{
  "recovery": {
    "before": {
      "rows": 75,
      "business_digest": "1321d375742d806a1f0fef82be9e2862af04f5d18f3a976792a6b1d9aa1b4383",
      "version": 0,
      "schema": [
        [
          "trip_id",
          "string"
        ],
        [
          "driver_id",
          "string"
        ],
        [
          "city",
          "string"
        ],
        [
          "start_utc",
          "timestamp"
        ],
        [
          "end_utc",
          "timestamp"
        ],
        [
          "trip_date_local",
          "date"
        ],
        [
          "fare_sar",
          "decimal(12,2)"
        ],
       

In [41]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day03_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day03_handoff.zip


# Lab 5 and 6 (Day 4)

In [42]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_lo_ncbvo


In [45]:
%pip install -q kafka-python==2.2.15

In [53]:
%%bash
set -e

KAFKA_VERSION="4.0.2"
SCALA_VERSION="2.13"

KAFKA_HOME="/content/kafka_${SCALA_VERSION}-${KAFKA_VERSION}"
KAFKA_TGZ="/content/kafka_${SCALA_VERSION}-${KAFKA_VERSION}.tgz"
KAFKA_CONFIG="/content/masar-kafka.properties"
KAFKA_DATA="/content/masar-kafka-data"
KAFKA_LOG="/content/masar-kafka.log"

echo "=== Java ==="
java -version

# Download Kafka only if it is not already downloaded
if [ ! -d "$KAFKA_HOME" ]; then
    echo "Downloading Kafka ${KAFKA_VERSION}..."

    wget -q \
      "https://archive.apache.org/dist/kafka/${KAFKA_VERSION}/kafka_${SCALA_VERSION}-${KAFKA_VERSION}.tgz" \
      -O "$KAFKA_TGZ"

    tar -xzf "$KAFKA_TGZ" -C /content
fi

echo "Kafka home: $KAFKA_HOME"

# Create a Colab-specific configuration
cp "$KAFKA_HOME/config/server.properties" "$KAFKA_CONFIG"

# Keep Kafka data inside /content
sed -i "s#^log.dirs=.*#log.dirs=$KAFKA_DATA#" "$KAFKA_CONFIG"

# Clean previous incomplete Kafka data
rm -rf "$KAFKA_DATA"

# Create KRaft cluster
KAFKA_CLUSTER_ID="$("$KAFKA_HOME/bin/kafka-storage.sh" random-uuid)"

echo "Cluster ID: $KAFKA_CLUSTER_ID"

"$KAFKA_HOME/bin/kafka-storage.sh" format \
    --standalone \
    -t "$KAFKA_CLUSTER_ID" \
    -c "$KAFKA_CONFIG"

# Start Kafka in background
nohup "$KAFKA_HOME/bin/kafka-server-start.sh" \
    "$KAFKA_CONFIG" \
    > "$KAFKA_LOG" 2>&1 &

echo "Kafka process started."

=== Java ===
Kafka home: /content/kafka_2.13-4.0.2
Cluster ID: 72jFXUcBSOaRiTjmdJbrhw
Formatting dynamic metadata voter directory /content/masar-kafka-data with metadata.version 4.0-IV3.
Kafka process started.


openjdk version "17.0.20" 2026-07-21
OpenJDK Runtime Environment (build 17.0.20+8-1-22.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20+8-1-22.04-Ubuntu, mixed mode, sharing)


In [54]:
import time
import socket

time.sleep(8)

def kafka_is_running():
    try:
        with socket.create_connection(
            ("127.0.0.1", 9092),
            timeout=3
        ):
            return True
    except OSError:
        return False

print("Kafka listening on 127.0.0.1:9092:", kafka_is_running())

Kafka listening on 127.0.0.1:9092: True


In [61]:
import json

from masar.streaming import stream_preflight

preflight = stream_preflight()

print(json.dumps(preflight, indent=2))

{
  "scope": "LOCAL_DEPENDENCY_CHECK_ONLY",
  "kafka_executed": false,
  "client_required": "2.2.15",
  "client_observed": "2.2.15",
  "bootstrap_servers": "127.0.0.1:9092",
  "tcp_listening": true,
  "issues": []
}


In [1]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "io.delta:delta-spark_2.12:3.3.3,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8 "
    "pyspark-shell"
)

print(os.environ["PYSPARK_SUBMIT_ARGS"])

--packages io.delta:delta-spark_2.12:3.3.3,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8 pyspark-shell


In [2]:
%pip install -q kafka-python==2.2.15

In [3]:
import importlib.metadata

print(importlib.metadata.version("kafka-python"))

2.2.15


In [5]:
from pathlib import Path

print(
    "Kafka installed:",
    Path("/content/kafka_2.13-4.0.2").exists()
)

Kafka installed: True


In [6]:
import subprocess

kafka_home = "/content/kafka_2.13-4.0.2"
config = "/content/masar-kafka.properties"
log_file = "/content/masar-kafka.log"

log = open(log_file, "w")

kafka_process = subprocess.Popen(
    [
        f"{kafka_home}/bin/kafka-server-start.sh",
        config,
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print("Kafka PID:", kafka_process.pid)

Kafka PID: 53187


In [7]:
import socket
import time

time.sleep(8)

try:
    with socket.create_connection(
        ("127.0.0.1", 9092),
        timeout=3,
    ):
        print("✅ Kafka broker running")
except OSError as e:
    print("❌ Kafka broker not running:", e)

✅ Kafka broker running


In [46]:
import socket
import time

time.sleep(2)

for port in [9092, 9093]:
    s = socket.socket()
    result = s.connect_ex(("127.0.0.1", port))
    s.close()
    print(port, "OPEN" if result == 0 else "CLOSED")

9092 OPEN
9093 OPEN


In [47]:
%%bash

KAFKA_CONFIG="/content/masar-kafka.properties"

sed -i \
's#^listeners=.*#listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093#' \
"$KAFKA_CONFIG"

sed -i \
's#^advertised.listeners=.*#advertised.listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093#' \
"$KAFKA_CONFIG"

sed -i \
's#^controller.quorum.bootstrap.servers=.*#controller.quorum.bootstrap.servers=127.0.0.1:9093#' \
"$KAFKA_CONFIG"

echo "Relevant Kafka configuration:"
grep -E \
'^(process.roles|node.id|listeners|advertised.listeners|controller.quorum.bootstrap.servers|controller.listener.names|log.dirs)=' \
"$KAFKA_CONFIG"

Relevant Kafka configuration:
process.roles=broker,controller
node.id=1
controller.quorum.bootstrap.servers=127.0.0.1:9093
listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093
advertised.listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093
controller.listener.names=CONTROLLER
log.dirs=/content/masar-kafka-data


In [48]:
%%bash
set -e

KAFKA_HOME="/content/kafka_2.13-4.0.2"
KAFKA_CONFIG="/content/masar-kafka.properties"
KAFKA_DATA="/content/masar-kafka-data"

rm -rf "$KAFKA_DATA"

CLUSTER_ID="$("$KAFKA_HOME/bin/kafka-storage.sh" random-uuid)"

echo "Cluster ID: $CLUSTER_ID"

"$KAFKA_HOME/bin/kafka-storage.sh" format \
    --standalone \
    -t "$CLUSTER_ID" \
    -c "$KAFKA_CONFIG"

Cluster ID: VC_oMClNQJO58AQpmLzNyA
Formatting dynamic metadata voter directory /content/masar-kafka-data with metadata.version 4.0-IV3.


In [49]:
import subprocess

kafka_home = "/content/kafka_2.13-4.0.2"
config = "/content/masar-kafka.properties"
log_file = "/content/masar-kafka.log"

log = open(log_file, "w")

kafka_process = subprocess.Popen(
    [
        f"{kafka_home}/bin/kafka-server-start.sh",
        config,
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print("Kafka PID:", kafka_process.pid)

Kafka PID: 72207


In [51]:
import socket

for port in [9092, 9093]:
    try:
        with socket.create_connection(
            ("127.0.0.1", port),
            timeout=3
        ):
            print(f"✅ 127.0.0.1:{port} reachable")
    except OSError as e:
        print(f"❌ 127.0.0.1:{port} unavailable:", e)

✅ 127.0.0.1:9092 reachable
✅ 127.0.0.1:9093 reachable


In [54]:
import subprocess
import os
import signal
import time
import socket
import shutil
import re
from pathlib import Path

KAFKA_HOME = Path("/content/kafka_2.13-4.0.2")
KAFKA_CONFIG = Path("/content/masar-kafka.properties")
KAFKA_DATA = Path("/content/masar-kafka-data")
KAFKA_LOG = Path("/content/masar-kafka.log")

# --------------------------------------------------
# 1. Kill ALL existing Kafka JVMs
# --------------------------------------------------

ps = subprocess.run(
    ["ps", "-eo", "pid,args"],
    capture_output=True,
    text=True
).stdout

for line in ps.splitlines():
    if "kafka.Kafka" in line or "KafkaRaftServer" in line:
        parts = line.strip().split(None, 1)

        if parts:
            pid = int(parts[0])
            print("Stopping old Kafka PID:", pid)

            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

time.sleep(3)

# --------------------------------------------------
# 2. Make sure ports are really closed
# --------------------------------------------------

for port in [9092, 9093]:
    s = socket.socket()
    result = s.connect_ex(("127.0.0.1", port))
    s.close()

    print(
        f"Port {port}:",
        "CLOSED ✅" if result != 0 else "STILL OPEN ❌"
    )

Stopping old Kafka PID: 72207
Port 9092: CLOSED ✅
Port 9093: CLOSED ✅


In [55]:
original_config = KAFKA_HOME / "config/server.properties"

shutil.copy(original_config, KAFKA_CONFIG)

config = KAFKA_CONFIG.read_text()

config = re.sub(
    r"^controller\.quorum\.bootstrap\.servers=.*$",
    "controller.quorum.bootstrap.servers=127.0.0.1:9093",
    config,
    flags=re.MULTILINE
)

config = re.sub(
    r"^listeners=.*$",
    "listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093",
    config,
    flags=re.MULTILINE
)

config = re.sub(
    r"^advertised\.listeners=.*$",
    "advertised.listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093",
    config,
    flags=re.MULTILINE
)

config = re.sub(
    r"^log\.dirs=.*$",
    "log.dirs=/content/masar-kafka-data",
    config,
    flags=re.MULTILINE
)

KAFKA_CONFIG.write_text(config)

print("Config ready:")
for line in config.splitlines():
    if line.startswith((
        "process.roles=",
        "node.id=",
        "controller.quorum.bootstrap.servers=",
        "listeners=",
        "advertised.listeners=",
        "controller.listener.names=",
        "log.dirs="
    )):
        print(line)

Config ready:
process.roles=broker,controller
node.id=1
controller.quorum.bootstrap.servers=127.0.0.1:9093
listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093
advertised.listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093
controller.listener.names=CONTROLLER
log.dirs=/content/masar-kafka-data


In [56]:
if KAFKA_DATA.exists():
    shutil.rmtree(KAFKA_DATA)

storage = str(KAFKA_HOME / "bin/kafka-storage.sh")

cluster_id = subprocess.check_output(
    [storage, "random-uuid"],
    text=True
).strip()

print("Cluster ID:", cluster_id)

subprocess.run(
    [
        storage,
        "format",
        "--standalone",
        "-t", cluster_id,
        "-c", str(KAFKA_CONFIG)
    ],
    check=True
)

print("✅ Kafka storage formatted")

Cluster ID: mEOyYIIsSYyNAddPD-0Tcw
✅ Kafka storage formatted


In [57]:
log_handle = open(KAFKA_LOG, "w")

kafka_process = subprocess.Popen(
    [
        str(KAFKA_HOME / "bin/kafka-server-start.sh"),
        str(KAFKA_CONFIG)
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT
)

print("Kafka PID:", kafka_process.pid)

Kafka PID: 75207


In [58]:
topics_cmd = str(KAFKA_HOME / "bin/kafka-topics.sh")

ready = False

for attempt in range(12):
    time.sleep(2)

    # Did Kafka crash?
    if kafka_process.poll() is not None:
        print("❌ Kafka process exited:", kafka_process.returncode)
        break

    try:
        result = subprocess.run(
            [
                topics_cmd,
                "--bootstrap-server",
                "127.0.0.1:9092",
                "--list"
            ],
            capture_output=True,
            text=True,
            timeout=8
        )

        if result.returncode == 0:
            ready = True
            print("✅ Kafka is fully ready")
            print("Topics:", result.stdout)
            break

        print(
            f"Attempt {attempt + 1}:",
            result.stderr[-300:]
        )

    except subprocess.TimeoutExpired:
        print(f"Attempt {attempt + 1}: Kafka not ready yet")

if not ready:
    print("\n❌ Kafka did not become healthy.")
    print("\nLast Kafka log lines:\n")

    print(
        "\n".join(
            KAFKA_LOG.read_text(
                errors="replace"
            ).splitlines()[-80:]
        )
    )

✅ Kafka is fully ready
Topics: 



In [59]:
import uuid

from kafka.admin import KafkaAdminClient, NewTopic

test_topic = "masar-day04-" + uuid.uuid4().hex

print("Creating:", test_topic)

admin = KafkaAdminClient(
    bootstrap_servers="127.0.0.1:9092",
    client_id="masar-admin-test",
    request_timeout_ms=15000,
    api_version_auto_timeout_ms=8000
)

try:
    admin.create_topics(
        [
            NewTopic(
                name=test_topic,
                num_partitions=2,
                replication_factor=1
            )
        ],
        timeout_ms=15000
    )

    print("✅ kafka-python create_topics works")

finally:
    admin.close()

Creating: masar-day04-b2317eee38bf47faa9aba9c43394f7fb
✅ kafka-python create_topics works


In [60]:
from masar.streaming import run_stream_lab
spark = start_spark(WORK, kafka=True)
try:
    result = run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result('lab05_streaming', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Transport rows:', [phase['transport_rows'] for phase in result['phases']])
    print('Unique event IDs:', [phase['unique_event_ids'] for phase in result['phases']])
    spark.read.format('delta').load(str(WORK/result['event_table'])).select('event_id','trip_id','event_ts').orderBy('event_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY04_NATIVE_STREAMING",
  "checks": {
    "phase_transport_counts": true,
    "phase_event_counts": true,
    "transport_keys_always_unique": true,
    "restart_same_query_identity": true,
    "restart_new_execution_ids": true,
    "checkpoint_same_for_all_phases": true,
    "actual_checkpoint_files_present": true,
    "producer_consumer_offsets_reconcile": true,
    "source_json_text_preserved": true,
    "event_content_matches_source": true,
    "unique_events_delta_readback": true,
    "all_events_link_to_trusted_trips": true,
    "late_event_retained": true
  }
}
Transport rows: [216, 216, 218, 219]
Unique event IDs: [216, 216, 216, 217]
+-----------+---------+-------------------------+
|event_id   |trip_id  |event_ts                 |
+-----------+---------+-------------------------+
|SYN_E0001_0|SYN_T0001|2026-06-01T06:00:00+03:00|
|SYN_E0001_1|SYN_T0001|2026-06-01T06:04:00+03:00|
|SYN_E0001_2|SYN_T0001|2026-06-01T06:08:00+03:00|
|SYN_E0002_0|SYN_T0002|2026-06-01T0

In [65]:
%pip install -q --upgrade \
    great-expectations==1.7.0 \
    pandas==2.2.3

In [66]:
from masar.quality_gate import run_quality_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_quality_lab(spark, SOURCE, WORK)
    validate_stage_result('lab06_quality', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Quarantined records:')
    spark.read.format('delta').load(str(WORK/result['quarantine_table'])).show(7, truncate=False)
    print('Approved rows:', spark.read.format('delta').load(str(WORK/result['approved_table'])).count())
finally:
    spark.stop()

Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

{
  "scope": "DAY04_NATIVE_QUALITY",
  "checks": {
    "trusted_and_rechecked_pass_gx": true,
    "mixed_candidate_fails_gx": true,
    "native_mixed_counts": true,
    "native_reasons_match_reference": true,
    "failed_candidate_not_promoted": true,
    "quarantine_delta_readback": true,
    "approved_readback_same_business_contents": true,
    "source_silver_untouched": true,
    "data_docs_exist_for_all_three_cases": true
  }
}
Quarantined records:
+-------------+----------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+---------------------------------------------------------------------+
|candidate_row|trip_id   |reason_codes       |raw_business_json                             

In [67]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day04_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day04_handoff.zip


# Lab 7 and 8 (Day 5)

In [68]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_lo_ncbvo


In [69]:
from masar.serving import run_recovery_exercise
spark = start_spark(WORK, kafka=False)
try:
    result = run_recovery_exercise(spark, SOURCE, WORK)
    validate_stage_result('lab07_gold_recovery', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
finally:
    spark.stop()

{
  "scope": "DAY05_NATIVE_RECOVERY",
  "checks": {
    "injected_failure_observed": true,
    "previous_release_preserved": true,
    "rebuild_has_new_identity": true,
    "content_equal": true
  }
}


In [70]:
from masar.serving import run_serving_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_serving_lab(spark, SOURCE, WORK)
    validate_stage_result('lab08_serving', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('BI totals:', json.dumps(result['bi_summary'], indent=2))
    from masar.serving import read_release
    _, observed_tables = read_release(spark, WORK)
    print('AI feature example:', observed_tables['ai.zone_hourly_features'][0])
    print('Future label example:', observed_tables['ai.zone_hourly_labels'][0])
finally:
    spark.stop()

{
  "scope": "DAY05_NATIVE_SERVING",
  "checks": {
    "gold.zone_hourly_demand_schema_and_keys": true,
    "gold.driver_daily_schema_and_keys": true,
    "bi.dim_zone_schema_and_keys": true,
    "bi.dim_driver_schema_and_keys": true,
    "bi.dim_date_schema_and_keys": true,
    "bi.fact_trips_schema_and_keys": true,
    "ai.zone_hourly_features_schema_and_keys": true,
    "ai.zone_hourly_labels_schema_and_keys": true,
    "fact_grain_75": true,
    "foreign_keys_valid": true,
    "gold_fact_totals_match": true,
    "events_aggregated_before_join": true,
    "group_grains_reconcile": true,
    "labels_not_fabricated": true,
    "feature_availability_checked": true,
    "feature_label_keys_aligned": true
  }
}
BI totals: [
  {
    "zone_key": "Z_DAMMAM",
    "trip_count": 25,
    "total_fare_sar": "670.40"
  },
  {
    "zone_key": "Z_JEDDAH",
    "trip_count": 25,
    "total_fare_sar": "625.20"
  },
  {
    "zone_key": "Z_RIYADH",
    "trip_count": 25,
    "total_fare_sar": "585.00"
  }

In [71]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day05_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day05_handoff.zip
